In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("../data/data_final.csv")

diagnosis_variables = []

with open("../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [4]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [5]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [6]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [ ]:
# {'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 200}
# Best C-index: 0.7380676257590892

sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf_1.split(X=x_train, y=y_train['event'])

'''best_model = RandomSurvivalForest(
    bootstrap = True, 
    max_depth = 10, 
    max_features = 'sqrt', 
    min_samples_leaf = 10,
    min_samples_split = 2, 
    n_estimators = 200,
    random_state=12)'''

param_grid = {
    'n_estimators': [100, 200, 500],    
    'max_depth': [5, 10, 15, None],    
    'max_features': ['sqrt', 'log2', 0.5],    
    'min_samples_leaf': [3, 10, 20],
}

In [ ]:
results_df, best_params, summary = generic_grid_search(
    RandomSurvivalForest,
    x_train,
    y_train,
    cv,
    param_grid,
    metrics=['c-index', 'c-index-ipcw', 'int-brier-score'],
    model_name='RSF',
    scale_data=False,
    scaler_type='standard',
    verbose=True,
    random_state=12,
    n_jobs=-1,
)


RSF GRID SEARCH (Manual Implementation)
Model: RandomSurvivalForest
Scaling: No
Metrics: ['c-index', 'c-index-ipcw', 'int-brier-score']
Parameter Grid:
  - n_estimators: [100, 200, 500]
  - max_depth: [5, 10, 15, None]
  - max_features: ['sqrt', 'log2', 0.5]
  - min_samples_leaf: [3, 10, 20]
Total combinations: 108

[1/108] Testing params: {'n_estimators': 100, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 3}
  c-index: 0.7994 ± 0.0727
  c-index-ipcw: 0.6815 ± 0.1271
  int-brier-score: 0.0092 ± 0.0020
[2/108] Testing params: {'n_estimators': 100, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 10}
  Elapsed: 0.1min, Est. remaining: 13.1min
  c-index: 0.8083 ± 0.0859
  c-index-ipcw: 0.6752 ± 0.1296
  int-brier-score: 0.0092 ± 0.0020
[3/108] Testing params: {'n_estimators': 100, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 20}
  Elapsed: 0.2min, Est. remaining: 12.8min
  c-index: 0.8032 ± 0.0973
  c-index-ipcw: 0.6818 ± 0.1182
  int-brier-score: 

In [9]:
# View the results DataFrame
results_df.sort_values(by='c-index_mean', ascending=False)

NameError: name 'results_df' is not defined

In [ ]:
results_df.sort_values(by='c-index_mean', ascending=False).to_csv("../results/rsf_grid_search_results.csv", index=False)

# Train and Test Final Model with Best Hyperparameters

In [6]:
# Load best hyperparameters from grid search results
best_results = pd.read_csv("../results/normal/rsf_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  n_estimators: {int(best_params['n_estimators'])}")
max_depth_val = best_params['max_depth']
if pd.isna(max_depth_val):
    print(f"  max_depth: None")
else:
    print(f"  max_depth: {int(max_depth_val)}")
print(f"  max_features: {best_params['max_features']}")
print(f"  min_samples_leaf: {int(best_params['min_samples_leaf'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  n_estimators: 100
  max_depth: 10
  max_features: 0.5
  min_samples_leaf: 3
  c-index-ipcw_mean (CV): 0.6913



In [7]:
# Train final model with best hyperparameters (NO SUBSAMPLING)
max_depth_val = best_params['max_depth']
max_depth = None if pd.isna(max_depth_val) else int(max_depth_val)

# Handle max_features - convert to proper type
max_features_val = best_params['max_features']
if isinstance(max_features_val, str):
    try:
        max_features = float(max_features_val)
    except ValueError:
        max_features = max_features_val
else:
    max_features = max_features_val

final_model = RandomSurvivalForest(
    n_estimators=int(best_params['n_estimators']),
    max_depth=max_depth,
    max_features=max_features,
    min_samples_leaf=int(best_params['min_samples_leaf']),
    random_state=12,
    n_jobs=-1
)

final_model.fit(x_train, y_train)
print("Model trained successfully on full training set (no subsampling)!")

Model trained successfully on full training set (no subsampling)!


In [8]:
# Test the model on test set
# Calculate c-index
c_index = final_model.score(x_test, y_test)

# Calculate c-index IPCW
tau = y_train["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train, y_test, final_model.predict(x_test), tau=tau)[0]

# Calculate integrated Brier score
surv_funcs = final_model.predict_survival_function(x_test)

# Use robust time filtering
max_time = min(y_train['time'].max(), y_test['time'].max()) * 0.9

# Use percentile-based times (5% to 90%)
times = np.percentile(y_train['time'], np.linspace(5, 90, 10))
# Ensure times are unique and sorted
times = np.unique(times)
# Filter times that are within valid range
times = times[(times > 0) & (times <= max_time)]

preds_at_times = np.array([[fn(t) for t in times] for fn in surv_funcs])
brier_score = integrated_brier_score(y_train, y_test, preds_at_times, times)

print("\nTest Set Performance (No Subsampling):")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print(f"  Integrated Brier Score: {brier_score:.4f}")


Test Set Performance (No Subsampling):
  C-index: 0.7894
  C-index IPCW: 0.7446
  Integrated Brier Score: 0.0081
